# 📓 NOTEBOOK THỬ NGHIỆM: RAG TỪ ZERO
> **Mục đích:** Thử nghiệm từng bước độc lập (Load -> Chunk -> Embed -> Retrieve -> Generate) mà không làm ảnh hưởng tới core code.

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Thêm đường dẫn src vào Python path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from rag.config import load_settings
from rag.loader import load_documents
from rag.chunker import chunk_documents, chunk_text
from rag.embeddings import LocalEmbedder
from rag.vector_store import VectorStore
from rag.prompts import SYSTEM_PROMPT, RAG_PROMPT_TEMPLATE, build_context
from rag.pipeline import RAGPipeline

settings = load_settings()
print("✅ Cấu hình hiện tại:")
print(f"- Data dir: {settings.raw_dir}")
print(f"- Chunk size: {settings.chunk_size}, Overlap: {settings.chunk_overlap}")
print(f"- Embedding model: {settings.embed_model}")

## 1. Thử nghiệm Đọc tài liệu (Loader)

In [ ]:
docs = load_documents(settings.raw_dir)
print(f"Đã đọc {len(docs)} tài liệu:\n")
for d in docs:
    print(f"📄 File: {d.source} ({len(d.text)} ký tự)")

## 2. Thử nghiệm Chia Chunk (Chunker)

In [ ]:
chunks = chunk_documents(docs, chunk_size=800, chunk_overlap=120)
print(f"Đã tạo tổng cộng {len(chunks)} chunks.\n")
if chunks:
    sample = chunks[0]
    print(f"--- [MẪU CHUNK ĐẦU TIÊN: {sample.source} (Index {sample.index})] ---")
    print(sample.text[:400] + "...")

## 3. Thử nghiệm Tạo Embedding Cục Bộ (Embedder)

In [ ]:
embedder = LocalEmbedder(settings.embed_model)
sample_vector = embedder.embed_query("RAG là gì?")
print(f"Vector shape: {sample_vector.shape}")
print(f"10 chiều đầu tiên: {sample_vector[:10]}")
print(f"Độ dài vector (L2 norm): {np.linalg.norm(sample_vector):.4f}")

## 4. Thử nghiệm Tìm kiếm Vector (Retrieve)

In [ ]:
store = VectorStore.load(settings.store_path)
query = "Muốn model biết dữ liệu mới mà không fine-tune thì làm gì?"
q_vec = embedder.embed_query(query)
results = store.search(q_vec, top_k=4)

print(f"🔍 Câu hỏi: {query}\n")
for i, r in enumerate(results, 1):
    print(f"[{i}] {r.source} (Điểm cosine: {r.score:.3f})")
    print(f"    Trích đoạn: {r.text[:150]}...\n")

## 5. Ghép Prompt Guardrail & Sinh câu trả lời qua Pipeline

In [ ]:
context = build_context(results)
full_prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=query)
print("=== PROMPT GỬI CHO LLM ===")
print(full_prompt[:600] + "...\n")

pipeline = RAGPipeline(settings)
answer = pipeline.answer(query)
print("=== PHẢN HỒI TỪ MÔ HÌNH QWEN3-4B ===")
print(answer.answer)